# Phase 2 — FinBERT 3-D Sentiment Ablation

Repeat the original Phase 2 pipeline and change only the text representation: 768-D FinBERT embedding → 3-D `[positive, neutral, negative]` FinBERT sentiment probabilities.

Expected original pipeline sizes: Train 15,969 rows / 15,534 windows; Validation 4,359 / 3,924; Test 6,275 / 5,840; window size 5. These are the documented project values.

**If the sanity checks fail, stop before training.**

## 1. Setup

## 2. Existing Phase 2 data pipeline

In [ ]:
from src.data.stocknet_dataset import load_and_clean,split_by_date,compute_norm_stats,normalize
print("✓ Existing project functions imported")

## 3. Exact Phase 2 parquet

In [ ]:
paths=glob.glob(os.path.join(REPO_ROOT,"**","*.parquet"),recursive=True)
paths=[p for p in paths if os.path.basename(p).lower()=="stocknet_final_modeling_set_phase2.parquet"]
if not paths: raise FileNotFoundError("Restore dataset/stocknet_final_modeling_set_phase2.parquet first.")
PHASE2_PATH=paths[0]
print(PHASE2_PATH)

## 4. Load, clean, and perform the original global split

In [ ]:
df=load_and_clean(PHASE2_PATH)
print("Cleaned rows:",len(df))
assert len(df)==26603, f"Expected 26603 rows, got {len(df)}"
train_df,val_df,test_df=split_by_date(df)
print(len(train_df),len(val_df),len(test_df))
assert (len(train_df),len(val_df),len(test_df))==(15969,4359,6275), "STOP: split mismatch"
print("✓ Original row split verified")

## 5. Original structured features

In [ ]:
PRICE_FEATURES=["Return","RSI_14","MACD","MACD_Signal","MACD_Hist","Volatility_5","Volatility_20","Price_MA5_Ratio","Price_MA10_Ratio","Price_MA20_Ratio","Volume_Change","HL_Spread","MA_5","MA_10"]
FUNDAMENTAL_FEATURES=["Revenue","NetIncome","TotalAssets","TotalLiabilities","StockholdersEquity","EPS","Cash","ROA"]
for c in PRICE_FEATURES+FUNDAMENTAL_FEATURES: assert c in df.columns,c
print(len(PRICE_FEATURES),len(FUNDAMENTAL_FEATURES))

## 6. FinBERT classification head: 3 sentiment probabilities

In [ ]:
MODEL_NAME="ProsusAI/finbert"
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME)
finbert=AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(DEVICE)
finbert.eval()
print(finbert.config.id2label)
assert finbert.config.num_labels==3

## 7. Generate sentiment from the existing Company_Texts rows

In [ ]:
assert "Company_Texts" in df.columns, "Company_Texts column missing"
texts=df["Company_Texts"].fillna("").astype(str).tolist()
sentiment=np.zeros((len(df),3),dtype=np.float32)
BATCH=32
with torch.no_grad():
    for start in range(0,len(texts),BATCH):
        batch=texts[start:start+BATCH]
        valid=[i for i,t in enumerate(batch) if t.strip()]
        if valid:
            inputs=tokenizer([batch[i] for i in valid],padding=True,truncation=True,max_length=256,return_tensors="pt")
            inputs={k:v.to(DEVICE) for k,v in inputs.items()}
            probs=torch.softmax(finbert(**inputs).logits,dim=-1).cpu().numpy().astype(np.float32)
            for j,pos in enumerate(valid): sentiment[start+pos]=probs[j]
        if start%(BATCH*20)==0: print(f"{min(start+BATCH,len(texts))}/{len(texts)}")
print(sentiment.shape,np.isnan(sentiment).sum(),np.isinf(sentiment).sum())
print(sentiment[:3],sentiment[:3].sum(axis=1))

## 8. Attach sentiment without changing row alignment

In [ ]:
df2=df.copy()
SENTIMENT_COLS=["FinBERT_Positive","FinBERT_Neutral","FinBERT_Negative"]
for i,c in enumerate(SENTIMENT_COLS): df2[c]=sentiment[:,i]
train2=df2.loc[train_df.index].copy(); val2=df2.loc[val_df.index].copy(); test2=df2.loc[test_df.index].copy()
assert (len(train2),len(val2),len(test2))==(15969,4359,6275)
print("✓ Exact row alignment preserved")

## 9. Train-only normalization

In [ ]:
STRUCTURED_COLS=PRICE_FEATURES+FUNDAMENTAL_FEATURES
means,stds=compute_norm_stats(train2,STRUCTURED_COLS)
train2=normalize(train2,STRUCTURED_COLS,means,stds)
val2=normalize(val2,STRUCTURED_COLS,means,stds)
test2=normalize(test2,STRUCTURED_COLS,means,stds)
print("✓ Normalization complete")

## 10. Same 5-day temporal windows

In [ ]:
class WindowDataset(Dataset):
    def __init__(self,frame,columns,window=5):
        frame=frame.sort_values(["Ticker","Date"]).reset_index(drop=True); self.samples=[]
        for _,g in frame.groupby("Ticker"):
            g=g.sort_values("Date").reset_index(drop=True)
            for i in range(window,len(g)):
                x=g.iloc[i-window:i][columns].to_numpy(np.float32); y=int(g.iloc[i]["Target"])
                self.samples.append((torch.tensor(x),torch.tensor(y,dtype=torch.long)))
    def __len__(self): return len(self.samples)
    def __getitem__(self,i): return self.samples[i]
WINDOW=5
ALL_FEATURES=STRUCTURED_COLS+SENTIMENT_COLS
train_full=WindowDataset(train2,ALL_FEATURES,WINDOW)
val_full=WindowDataset(val2,ALL_FEATURES,WINDOW)
test_full=WindowDataset(test2,ALL_FEATURES,WINDOW)
counts=(len(train_full),len(val_full),len(test_full))
print(counts)
assert counts==(15534,3924,5840), f"STOP: expected (15534,3924,5840), got {counts}"
print("✓ Original window counts verified")

## 11. Four ablations

In [ ]:
EXPERIMENTS={
"A_Price":PRICE_FEATURES,
"B_Price_Fundamentals":PRICE_FEATURES+FUNDAMENTAL_FEATURES,
"C_Price_FinBERT_Sentiment":PRICE_FEATURES+SENTIMENT_COLS,
"D_Price_Fundamentals_FinBERT_Sentiment":ALL_FEATURES}
print({k:len(v) for k,v in EXPERIMENTS.items()})

## 12. Select columns from the same windows

In [ ]:
class SelectedDataset(Dataset):
    def __init__(self,full,all_cols,selected): self.full=full; self.idx=[all_cols.index(c) for c in selected]
    def __len__(self): return len(self.full)
    def __getitem__(self,i):
        x,y=self.full[i]; return x[:,self.idx],y
datasets={}
for name,cols in EXPERIMENTS.items():
    datasets[name]={"train":SelectedDataset(train_full,ALL_FEATURES,cols),"val":SelectedDataset(val_full,ALL_FEATURES,cols),"test":SelectedDataset(test_full,ALL_FEATURES,cols)}

## 13. Same LSTM training protocol

In [ ]:
class LSTMBaseline(nn.Module):
    def __init__(self,input_dim,hidden=64,layers=2,dropout=.2):
        super().__init__(); self.lstm=nn.LSTM(input_dim,hidden,layers,batch_first=True,dropout=dropout); self.head=nn.Sequential(nn.Linear(hidden,32),nn.ReLU(),nn.Dropout(dropout),nn.Linear(32,2))
    def forward(self,x):
        _,(h,_) = self.lstm(x); return self.head(h[-1])

def evaluate(model,loader):
    model.eval(); yt=[];yp=[];pr=[]
    with torch.no_grad():
        for x,y in loader:
            logits=model(x.to(DEVICE)); yt+=y.numpy().tolist(); yp+=logits.argmax(1).cpu().numpy().tolist(); pr+=torch.softmax(logits,1)[:,1].cpu().numpy().tolist()
    yt=np.asarray(yt); yp=np.asarray(yp); pr=np.asarray(pr)
    return {"accuracy":accuracy_score(yt,yp),"f1":f1_score(yt,yp,zero_division=0),"mcc":matthews_corrcoef(yt,yp),"auc":roc_auc_score(yt,pr)}

def train_one(bundle,input_dim,max_epochs=50,patience=10):
    tr=DataLoader(bundle["train"],batch_size=64,shuffle=True); va=DataLoader(bundle["val"],batch_size=64,shuffle=False); te=DataLoader(bundle["test"],batch_size=64,shuffle=False)
    model=LSTMBaseline(input_dim).to(DEVICE); opt=torch.optim.Adam(model.parameters(),lr=.001,weight_decay=1e-4); loss_fn=nn.CrossEntropyLoss(); best=-float("inf"); state=None; wait=0
    for ep in range(1,max_epochs+1):
        model.train()
        for x,y in tr:
            x=x.to(DEVICE); y=y.to(DEVICE); opt.zero_grad(); loss=loss_fn(model(x),y); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
        vm=evaluate(model,va)
        if vm["mcc"]>best:
            best=vm["mcc"]; state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; wait=0
        else: wait+=1
        if ep==1 or ep%5==0: print(f"Ep {ep:3d} | Val Acc={vm['accuracy']:.3f} MCC={vm['mcc']:.4f} AUC={vm['auc']:.4f}")
        if wait>=patience: print("Early stop at epoch",ep); break
    model.load_state_dict(state); return model,evaluate(model,va),evaluate(model,te)

## 14. Run all four

In [ ]:
results={}; models={}
for name,cols in EXPERIMENTS.items():
    print("\n"+"="*80+"\n"+name+"\n"+"="*80); set_seed()
    model,val,test=train_one(datasets[name],len(cols)); models[name]=model; results[name]={"validation":val,"test":test}; print("Validation:",val); print("Test:",test)

## 15. Results and validation-based selection

In [ ]:
rows=[]
for n,r in results.items(): rows.append({"Experiment":n,**{f"Val_{k.title()}":v for k,v in r["validation"].items()},**{f"Test_{k.title()}":v for k,v in r["test"].items()}})
results_df=pd.DataFrame(rows).sort_values(["Val_Mcc","Val_Auc"],ascending=False)
display(results_df)
ranking=sorted(results.items(),key=lambda z:(z[1]["validation"]["mcc"],z[1]["validation"]["auc"]),reverse=True)
BEST=ranking[0][0]
print("Selected:",BEST)
print("Final test:",results[BEST]["test"])

## 16. Previous 768-D FinBERT references

| Feature Set | Accuracy | F1 | MCC | AUC |
|---|---:|---:|---:|---:|
| Price + Company FinBERT | 0.5248 | 0.4721 | 0.0459 | 0.5304 |
| Price + Fundamentals + Company FinBERT | 0.4870 | 0.6550 | 0.0000 | 0.5270 |

This comparison is meaningful only because the split, normalization, windowing, and training protocol are kept consistent.

## 17. Save results and generate the `.md` report

In [ ]:
RESULT_DIR=os.path.join(REPO_ROOT,"results","phase2_finbert_sentiment"); os.makedirs(RESULT_DIR,exist_ok=True)
results_df.to_csv(os.path.join(RESULT_DIR,"results.csv"),index=False)
with open(os.path.join(RESULT_DIR,"results.json"),"w") as f: json.dump(results,f,indent=2)
report=["# Phase 2 — FinBERT 3-D Sentiment Ablation","","## Objective","","Repeat the original Phase 2 pipeline while replacing the 768-D Company FinBERT representation with three FinBERT sentiment probabilities.","","## Verified pipeline","","- Train rows: 15,969","- Validation rows: 4,359","- Test rows: 6,275","- Train windows: 15,534","- Validation windows: 3,924","- Test windows: 5,840","- Window size: 5","","## Results","",results_df.to_markdown(index=False),"","## Selected configuration",f"**{BEST}** selected using validation MCC, then validation AUC.","","## Final test result","", "| Metric | Value |","|---|---:|"]
for k,v in results[BEST]["test"].items(): report.append(f"| {k.upper()} | {v:.4f} |")
report += ["","## Previous 768-D references","","| Feature Set | Accuracy | F1 | MCC | AUC |","|---|---:|---:|---:|---:|","| Price + Company FinBERT | 0.5248 | 0.4721 | 0.0459 | 0.5304 |","| Price + Fundamentals + Company FinBERT | 0.4870 | 0.6550 | 0.0000 | 0.5270 |"]
Path(os.path.join(RESULT_DIR,"phase2_finbert_sentiment.md")).write_text("\n".join(report),encoding="utf-8")
print(RESULT_DIR)

## 18. Final sanity check

Do not use the results unless the notebook verified exactly:

```text
Train rows: 15969
Val rows: 4359
Test rows: 6275
Train windows: 15534
Val windows: 3924
Test windows: 5840
Window size: 5
```